# Week 11: Advanced Specialized Machine Learning
## Notebook 09: Survival Analysis: Kaplan-Meier Estimator & Cox Hazards

### 1. Conceptual Intuition & Theoretical Background
In customer churn, predictive maintenance, and clinical trials, the target variable is **Time-to-Event** $T$. Standard regression fails because observations are **right-censored**: the study ends before an event occurs (e.g. a customer is still active at month 12).

**1. Kaplan-Meier Non-Parametric Estimator:**
Estimates the survival function $S(t) = P(T > t)$ across distinct event times $t_1 < t_2 < \dots < t_k$:
$$\hat{S}(t) = \prod_{t_i \le t} \left( 1 - \frac{d_i}{n_i} \right)$$
Where $d_i$ is the number of events (churns/deaths) at $t_i$, and $n_i$ is the number of individuals at risk right before $t_i$.

**2. Cox Proportional Hazards Model:**
Semi-parametric model for the hazard rate $\lambda(t \mid X)$:
$$\lambda(t \mid X) = \lambda_0(t) \exp(X \beta)$$
Where $\lambda_0(t)$ is an arbitrary baseline hazard function, and $\exp(\beta_j)$ is the Hazard Ratio (HR) for feature $j$.

---

### 2. Scratch Python Implementation: Kaplan-Meier Estimator

In [ ]:
import numpy as np
import pandas as pd

def kaplan_meier_scratch(durations, event_observed):
    """
    Computes Kaplan-Meier survival curves handling right-censored data.
    durations: array of observed event or censoring times
    event_observed: 1 if event occurred, 0 if censored
    """
    df = pd.DataFrame({'time': durations, 'event': event_observed})
    
    # Sort by time ascending
    df = df.sort_values('time').reset_index(drop=True)
    
    # Extract distinct event timestamps
    unique_times = np.sort(df[df['event'] == 1]['time'].unique())
    
    survival_probs = [1.0]
    timeline = [0.0]
    
    current_surv = 1.0
    for t in unique_times:
        # At risk: individuals with duration >= t
        n_at_risk = np.sum(df['time'] >= t)
        # Number of events at exactly time t
        d_events = np.sum((df['time'] == t) & (df['event'] == 1))
        
        if n_at_risk > 0:
            current_surv *= (1.0 - (d_events / n_at_risk))
            
        timeline.append(t)
        survival_probs.append(current_surv)
        
    return np.array(timeline), np.array(survival_probs)

# Simulate SaaS Customer Subscription Durations (Months)
np.random.seed(42)
n_users = 200

# True churn event times ~ Exponential(scale=18 months)
true_times = np.random.exponential(scale=18.0, size=n_users)
# End of observation study at Month 12 (Right-censoring)
censoring_cutoff = 12.0

observed_duration = np.minimum(true_times, censoring_cutoff)
event_occurred = (true_times <= censoring_cutoff).astype(int)

t_line, s_curve = kaplan_meier_scratch(observed_duration, event_occurred)

# Median Survival Time (where S(t) = 0.5)
median_idx = np.searchsorted(-s_curve, -0.5)
median_churn_time = t_line[median_idx] if median_idx < len(t_line) else "> 12 months"

print("--- Kaplan-Meier Survival Analysis ---")
print(f"Total Users Tracked:      {n_users}")
print(f"Observed Churn Events:    {np.sum(event_occurred)} ({np.mean(event_occurred)*100:.1f}%)")
print(f"Censored Active Users:    {np.sum(event_occurred == 0)} ({np.mean(event_occurred==0)*100:.1f}%)")
print(f"6-Month Survival Rate:    {s_curve[np.searchsorted(t_line, 6.0)]*100:.1f}%")
print(f"12-Month Survival Rate:   {s_curve[-1]*100:.1f}%")
print(f"Estimated Median Churn:   {median_churn_time}")

### 3. Step-by-Step Code Walkthrough
1. **At-Risk Tracking ($n_i$):** Accounts for censored individuals up to their departure without skewing event ratios.
2. **Product-Limit Formulation:** Multiplies consecutive conditional survival fractions $(1 - d_i / n_i)$.
3. **Censoring Accommodation:** Handles customers still subscribed at study completion without dropping them or assuming immediate churn.

---

### 4. Industry Applications & Trade-offs
- **SaaS Customer Lifetime Value (LTV):** Forecasting expected churn milestones for revenue planning.
- **Trade-off:** Standard Cox models assume the Proportional Hazards assumption (constant hazard ratio over time).

---

### 5. Top Interview Questions & Answers
**Q1: Why does standard linear regression fail when predicting time-to-churn on customer datasets?**
> **Answer:** Linear regression cannot accommodate right-censoring: excluding active customers introduces survival bias, while treating censoring time as event time underestimates true lifespan.

**Q2: What is the Hazard Ratio (HR) in a Cox Proportional Hazards model?**
> **Answer:** $\text{HR} = \exp(\beta)$. An HR of $1.5$ indicates that an applicant with that feature has a $50\%$ higher probability of experiencing the event at any given time step compared to baseline.